# Bound one decode step
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week6/Decode_Step_Bound_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 6, Module 1: Roofline analysis**

Every operation has two lower bounds on its time. The arithmetic cannot finish faster than its FLOPs divided by peak compute, and the data cannot arrive faster than its bytes divided by bandwidth. On a machine that overlaps the two, the larger bound sets the time. Dividing FLOPs by that time gives the roofline: attainable performance is the smaller of peak compute and bandwidth times arithmetic intensity.

The lecture works this out for one 4096-wide projection and for one decode step of Llama 3.1 8B on an H100. In this notebook you do the same on the T4: compute both bounds for the projection at decode and prefill sizes, measure the real time, place the points on a roofline, and bound a full decode step of Qwen 2.5 0.5B. You also see the difference between the distance to the roof and the distance to peak.

**What you will do**
1. Measure the T4's achieved compute and bandwidth.
2. Compute both bounds for a 4096 by 4096 projection at batch 1 and at 2048 tokens.
3. Measure the projection at several batch sizes and place the points on a roofline.
4. Bound one decode step of Qwen 2.5 0.5B and compare with its measured GPU time.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. The notebook takes about two minutes.

## 1. Measured roofs
Published peaks are the starting point, but kernels run below them. The cell measures the achieved FP16 matmul throughput as the best of three large square matrix multiplies, and the achieved bandwidth with a copy. Lab 6 does this in more detail.

In [ ]:
import statistics
import time
import torch

assert torch.cuda.is_available(), "This notebook needs a GPU: Runtime > Change runtime type > T4 GPU"
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

SPEC_FLOPS, SPEC_BW = 65e12, 320e9            # published T4 FP16 tensor-core peak and GDDR6 bandwidth


def median_seconds(fn, iters=30):
    for _ in range(5):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


peak_flops = 0.0
for n in (2048, 4096, 8192):
    a = torch.randn(n, n, device="cuda", dtype=torch.float16)
    peak_flops = max(peak_flops, 2 * n ** 3 / median_seconds(lambda: a @ a, iters=10))
    del a
src = torch.empty(256 * 2**20, dtype=torch.uint8, device="cuda")
dst = torch.empty_like(src)
peak_bw = 2 * src.numel() / median_seconds(lambda: dst.copy_(src))
del src, dst

print(f"Compute: {peak_flops / 1e12:5.1f} TFLOPS measured, {SPEC_FLOPS / 1e12:.0f} published")
print(f"Bandwidth: {peak_bw / 1e9:5.0f} GB/s measured, {SPEC_BW / 1e9:.0f} published")
print(f"Ridge point: {peak_flops / peak_bw:5.0f} FLOPs/byte measured, {SPEC_FLOPS / SPEC_BW:.0f} published")

**Reading the output**
- Both measured roofs are below the published ones, the compute roof by more. The measured ridge point is therefore lower than the published one: the measured roofs make more workloads compute-bound than the datasheet suggests.
- The rest of the notebook uses the measured roofs.

## 2. Two bounds for one projection
The lecture's example: one projection of a 4096-wide layer, a 4096 by 4096 weight matrix, in FP16.

- FLOPs: `2 x M x K x N`
- Bytes: `2 x (M x K + K x N + M x N)`, reading both inputs and writing the output once

During decode, `M` is 1: one token. During prefill of a 2048-token prompt, `M` is 2048.

In [ ]:
K = N = 4096


def bounds(M):
    flops = 2 * M * K * N
    nbytes = 2 * (M * K + K * N + M * N)
    t_compute = flops / peak_flops
    t_memory = nbytes / peak_bw
    return flops, nbytes, flops / nbytes, t_compute, t_memory


for label, M in (("decode, 1 token", 1), ("prefill, 2048 tokens", 2048)):
    flops, nbytes, ai, tc, tm = bounds(M)
    limit = "memory" if tm > tc else "compute"
    print(f"{label}: {flops / 1e6:,.0f} MFLOP, {nbytes / 1e6:.1f} MB, intensity {ai:,.1f} FLOPs/byte")
    print(f"   compute bound {tc * 1e6:8.1f} us, memory bound {tm * 1e6:8.1f} us -> limited by {limit}")

**Reading the output**
- For decode, the intensity is about 1 FLOP per byte. The memory bound is hundreds of times larger than the compute bound, so faster arithmetic would change nothing. The bytes are almost all the weight matrix.
- For prefill, the intensity is about a thousand. The weight is still read once, but it is used by 2048 tokens, so the compute bound is now the larger one.
- The same layer has two very different intensities, depending on how many tokens share each weight read.

## 3. Measured points on the roofline
The cell times the same projection for `M` from 1 to 2048 and computes the achieved FLOP rate. The roof at each point is `min(peak, bandwidth x intensity)`. Two ratios are printed:

- **of roof:** how close the kernel is to the best it could do at its intensity.
- **of peak:** how close it is to the machine's peak compute.

In [ ]:
w = torch.randn(K, N, device="cuda", dtype=torch.float16)
points, rows = [], []
for M in (1, 8, 32, 128, 512, 2048):
    x = torch.randn(M, K, device="cuda", dtype=torch.float16)
    flops, nbytes, ai, _, _ = bounds(M)
    sec = median_seconds(lambda: x @ w)
    points.append((ai, flops / sec))
    rows.append((M, ai, sec))

# The T4 is limited to 70 W. Long runs of large matmuls lower its clock, so a shorter
# kernel can run faster than the square matmuls of section 1. Use the best rate seen.
if max(p[1] for p in points) > peak_flops:
    peak_flops = max(p[1] for p in points)
    print(f"Compute roof raised to the best rate observed: {peak_flops / 1e12:.1f} TFLOPS\n")

print(f"{'M':>5} {'intensity':>10} {'time':>9} {'achieved':>11} {'of roof':>8} {'of peak':>8}")
for (M, ai, sec), (_, achieved) in zip(rows, points):
    roof = min(peak_flops, peak_bw * ai)
    print(f"{M:5d} {ai:10.1f} {sec * 1e6:7.1f} us {achieved / 1e12:7.2f} TF/s {achieved / roof:8.0%} {achieved / peak_flops:8.1%}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ai = np.logspace(-0.5, 4, 200)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.loglog(ai, np.minimum(peak_flops, peak_bw * ai) / 1e12, "k-", label="measured roof")
ax.loglog(ai, np.minimum(SPEC_FLOPS, SPEC_BW * ai) / 1e12, "k:", label="published roof")
ax.loglog([p[0] for p in points], [p[1] / 1e12 for p in points], "o", color="0.4", label="projection, M = 1 to 2048")
for (x_ai, y), M in zip(points, (1, 8, 32, 128, 512, 2048)):
    ax.annotate(f"M={M}", (x_ai, y / 1e12), textcoords="offset points", xytext=(4, -10), fontsize=7)
ax.axvline(peak_flops / peak_bw, color="0.6", linestyle="--", linewidth=0.8)
ax.set_xlabel("arithmetic intensity (FLOPs per byte)")
ax.set_ylabel("TFLOPS")
ax.legend(frameon=False, fontsize=8)
ax.set_title("One 4096 x 4096 projection on a T4")
plt.tight_layout()
plt.show()

**Reading the output**
- At `M = 1` the point sits on the sloped part of the roof, at a tiny fraction of peak. That is not a bad kernel. It is the roof at that intensity, the lecture's "decode can reach at most one 295th of peak" on an H100.
- As `M` grows, the points climb the slope, because each weight read now serves more tokens. Batching is the lever for decode.
- The largest case, `M = 2048`, can fall back below the roof. It runs long enough for the T4 to lower its clock under its 70 W power limit, the same effect seen in section 1.
- The **of roof** column says how good the kernel is. The **of peak** column says how good the workload's intensity is. A decode kernel can be close to its roof and still far below peak, and the only way up is to move right.

## 4. Bound a whole decode step
For a full model the same counting applies. At batch 1 every parameter is read once per step and used for about two FLOPs.

The cell loads Qwen 2.5 0.5B, computes both bounds for one decode step, and compares them with the GPU time of a real step from the profiler.

In [ ]:
from torch.profiler import profile, ProfilerActivity
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float16).cuda().eval()
n_params = sum(p.numel() for p in model.parameters())
weight_bytes = 2 * n_params
t_compute = 2 * n_params / peak_flops
t_memory = weight_bytes / peak_bw
print(f"{n_params / 1e6:.0f} M parameters, {weight_bytes / 1e9:.2f} GB of weights")
print(f"compute bound {t_compute * 1e3:.3f} ms, memory bound {t_memory * 1e3:.2f} ms per step")

with torch.inference_mode():
    out = model(input_ids=torch.randint(0, 50000, (1, 64), device="cuda"), use_cache=True)
    state = {"cache": out.past_key_values, "tok": out.logits[:, -1:].argmax(-1)}

    def step():
        o = model(input_ids=state["tok"], past_key_values=state["cache"], use_cache=True)
        state["cache"], state["tok"] = o.past_key_values, o.logits[:, -1:].argmax(-1)

    for _ in range(3):
        step()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(5):
            step()
        torch.cuda.synchronize()
    wall = median_seconds(step, iters=10)

def self_device_us(e):
    # Newer PyTorch names this field self_device_time_total; older versions use self_cuda_time_total.
    us = getattr(e, "self_device_time_total", None)
    return us if us is not None else getattr(e, "self_cuda_time_total", 0)


gpu = sum(self_device_us(e) for e in prof.key_averages()
          if e.device_type == torch.autograd.DeviceType.CPU) / 5 / 1e6
print(f"measured GPU time per step: {gpu * 1e3:.2f} ms ({t_memory / gpu:.0%} of the bound's speed)")
print(f"measured wall time per step: {wall * 1e3:.2f} ms")

**Reading the output**
- The compute bound is a few hundredths of a millisecond. The memory bound is a few milliseconds. The step is memory-bound by about two orders of magnitude, as in the lecture's Llama example: 16 GB of weights on an H100 gives about 4.8 ms, and here about 1 GB on a T4 gives a few milliseconds.
- The measured GPU time is above the bound by a factor of two to three. The remaining kernels, the KV cache reads and launch gaps between dependent kernels add to it.
- The wall time is larger again. That is the overhead term, time when neither compute nor memory is busy, which Module 3 and the decode-loop demo in this folder analyze.

## Same notebook on other GPUs
The same notebook was run on four NVIDIA GPUs: the T4 from Colab's free tier, the L4, the A100 and the H100. The lecture marks values like these as MEASURED, observed on hardware, and SNAPSHOT, valid for one software release. Read the table for the pattern across GPUs, not for exact numbers.

<!-- gpu-table:start -->
| GPU | measured TFLOPS | measured GB/s | measured ridge | projection M=1, of roof | step bound, ms | step GPU time, ms | step wall time, ms |
|---|---|---|---|---|---|---|---|
| T4 (cc 7.5) | 33 | 227 | 143 | 89% | 4.36 | 8.98 | 29.1 |
| L4 (cc 8.9) | 78 | 229 | 340 | 221% | 4.32 | 5.87 | 45.3 |
| A100-80GB (cc 8.0) | 238 | 1,606 | 148 | 44% | 0.62 | 4.67 | 27.7 |
| H100 (cc 9.0) | 748 | 2,844 | 263 | 33% | 0.35 | 2.20 | 72.0 |

*MEASURED on cloud GPUs on 2026-11-18, PyTorch 2.15.0. SNAPSHOT: these values change with library versions and with the host CPU, and your Colab T4 can differ from the T4 row.*
<!-- gpu-table:end -->

**What the comparison shows**
- Each GPU has its own ridge point. The L4 has far more compute than the T4 but about the same memory bandwidth, so its ridge is much further to the right. The H100's measured ridge is close to the lecture's 295 FLOPs per byte, which comes from its published peaks.
- On the L4, the batch-1 projection lands above 100 percent of its roof. Its 32 MB weight matrix fits in the L4's 48 MB L2 cache, so the repeated runs read it from cache, not from DRAM. The roof depends on which memory level you count, which is the subject of Module 2.
- On the A100 and H100, the batch-1 projection reaches only part of the DRAM roof. One matrix-vector product takes a few tens of microseconds there, too short to reach full bandwidth.
- The full decode step shows the same pattern as the Week 4 decode demo: the faster the GPU, the further the GPU time is from the bandwidth bound, and the wall time is set by the host.

## Try this
1. In section 3, use FP32 tensors. Where do the points move on the plot, and why?
2. Using the lecture's H100 numbers (989 TFLOPS, 3.35 TB/s), recompute the bounds in section 2 and check that you get about 34 ns and 10 us for decode.
3. In section 4, estimate the batch size at which the decode step would become compute-bound on this T4.

## Summary
- Each operation has a compute bound and a memory bound. The larger one sets the time.
- The roofline is `min(peak, bandwidth x intensity)`. The ridge, where the two meet, belongs to the chip. The intensity belongs to the workload.
- Decode at batch 1 is on the sloped part of the roof at about 1 FLOP per byte. Batching moves it to the right.
- Distance to the roof measures the kernel. Distance to peak measures the workload's intensity.

Lecture: Week 6, Module 1, slides 9 to 14 (FLOPs and bytes, two lower bounds, the roofline, bounding one decode step, two distances).